# Camada Gold: Dimensões de Motivos e Canais
Objetivos desta etapa:
- Consumir os dados tratados da camada Silver;
- Disponibilizar as dimensões dim_motivo e dim_canal com os atributos necessários para análises e relacionamento com tabelas fato;
- Preservar os identificadores e atributos padronizados nas etapas anteriores;
- Gravar as dimensões em Delta Lake de forma idempotente, reconstruindo as tabelas a cada execução;
- Conferir a quantidade de registros, a amostra dos dados e possíveis identificadores duplicados.

## 1. Configuração

In [0]:
from pyspark.sql import functions as F

catalog = "vcredit"
silver_schema_name = "silver"
gold_schema_name = "gold"

silver_schema = f"{catalog}.{silver_schema_name}"
gold_schema = f"{catalog}.{gold_schema_name}"

spark.sql(f"CREATE SCHEMA IF NOT EXISTS {gold_schema}")

print(f"Lendo da Silver: {silver_schema}")
print(f"Gravando na Gold: {gold_schema}")

## 2. Dimensão Motivo
### Regra de negócio e estrutura
A dimensão de motivos disponibiliza os atributos usados para classificar os motivos de atendimento:
- id_motivo: identificador do motivo e chave da dimensão;
- nome_motivo: descrição do motivo;
- categoria: categoria padronizada na Silver;
- criticidade: nível de criticidade padronizado na Silver.
A etapa Silver já converte o identificador para inteiro, remove espaços extras, trata categorias ausentes conhecidas e padroniza acentuação e criticidade.

In [0]:
df_silver_motivos = spark.table(f"{silver_schema}.tb_motivos")

df_gold_motivo = (
    df_silver_motivos
    .select(
        F.col("id_motivo"),
        F.col("nome_motivo"),
        F.col("categoria"),
        F.col("criticidade")
    )
)
(
    df_gold_motivo.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", True)
    .saveAsTable(f"{gold_schema}.dim_motivo")
)

print(f"Tabela {gold_schema}.dim_motivo salva com sucesso!")
print(f"Total de motivos na dimensão: {df_gold_motivo.count()}")

display(df_gold_motivo.orderBy("id_motivo"))

## 3. Dimensão Canal
###Regra de negócio e estrutura
A dimensão de canais disponibiliza os atributos de identificação e classificação dos canais:
- id_canal: identificador do canal e chave da dimensão;
- nome_canal: nome do canal;
- tipo_canal: tipo ou classificação do canal;
- status_canal: status cadastral do canal.
A dimensão utiliza a tabela de canais já tratada na Silver. A seleção dos atributos mantém o escopo definido para a dimensão e evita repetir nesta etapa as transformações de limpeza realizadas anteriormente.

In [0]:
df_silver_canais = spark.table(f"{silver_schema}.canais")

df_gold_canal = (
    df_silver_canais
    .select(
        F.col("id_canal"),
        F.col("nome_canal"),
        F.col("tipo_canal"),
        F.col("status_canal")
    )
)
(
    df_gold_canal.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", True)
    .saveAsTable(f"{gold_schema}.dim_canal")
)

print(f"Tabela {gold_schema}.dim_canal salva com sucesso!")
print(f"Total de canais na dimensão: {df_gold_canal.count()}")

display(df_gold_canal.orderBy("id_canal"))